In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, glob, time
import numpy as np
import pandas as pd
import cv2

!pip install -q mediapipe
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

BASE = '/content/drive/MyDrive/Ketastasia/data'
CKPT_DIR = f'{BASE}/pipeline2_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

manifest_df = pd.read_csv(f'{BASE}/combined.csv')
split_map = dict(zip(manifest_df['video_id'], manifest_df['split']))
print(f"Loaded manifest: {len(manifest_df)} videos")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 13.3 MB/s eta 0:00:00
Loaded manifest: 1813 videos


In [2]:
if not os.path.exists('/content/Penn_Action'):
    print("Penn Action not found — extracting...")
    !tar -xzf '/content/drive/MyDrive/Ketastasia/data/Penn_Action.tar.gz' -C '/content/'
else:
    print("Penn Action already extracted")

if not os.path.exists('/content/Kaggle'):
    print("Kaggle not found — extracting...")
    !unzip -o -q '/content/drive/MyDrive/Ketastasia/data/kaggle.zip' -d '/content/Kaggle'
else:
    print("Kaggle already extracted")

Penn Action not found — extracting...
Kaggle not found — extracting...


In [3]:
model_path = "/content/pose_landmarker_full.task"
if not os.path.exists(model_path):
    import urllib.request
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
        "pose_landmarker_full/float16/1/pose_landmarker_full.task",
        model_path
    )

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE
)
detector = vision.PoseLandmarker.create_from_options(options)
print("33-keypoint detector ready")

33-keypoint detector ready


In [4]:
def resolve_video_source(row):
    """Returns ('frames', folder_path) for Penn Action, or ('video', file_path) for Kaggle."""
    if row['source'] == 'penn':
        frame_dir = f"/content/Penn_Action/frames/{row['video_id']}"
        return ('frames', frame_dir)
    else:
        for root, dirs, files in os.walk('/content/Kaggle'):
            for f in files:
                if f.rsplit('.', 1)[0] == row['video_id']:
                    return ('video', os.path.join(root, f))
        return (None, None)

In [5]:
STRIDE = 3  # extraction-time stride, ერთნაირად ორივე წყაროზე

def extract_landmarks_33(image_rgb):
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
    result = detector.detect(mp_image)
    if not result.pose_landmarks:
        return None
    return result.pose_landmarks[0]

def landmarks_to_row(row, frame_idx, lm):
    out = {
        'video_id': row['video_id'], 'source': row['source'],
        'frame': frame_idx, 'label': row['label'],
        'split': split_map[row['video_id']],
    }
    for j in range(33):
        out[f'x{j}'] = lm[j].x
        out[f'y{j}'] = lm[j].y
        out[f'vis{j}'] = lm[j].visibility
    return out

def process_video(row):
    kind, path = resolve_video_source(row)
    if kind is None:
        return []

    rows_out, frame_idx = [], 0

    if kind == 'video':
        cap = cv2.VideoCapture(path)
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % STRIDE == 0:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                lm = extract_landmarks_33(rgb)
                if lm:
                    rows_out.append(landmarks_to_row(row, frame_idx, lm))
            frame_idx += 1
        cap.release()
    else:
        frame_files = sorted(glob.glob(f'{path}/*.jpg'))
        for i, ff in enumerate(frame_files):
            if i % STRIDE == 0:
                img = cv2.imread(ff)
                rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                lm = extract_landmarks_33(rgb)
                if lm:
                    rows_out.append(landmarks_to_row(row, i, lm))

    return rows_out

In [8]:
import glob, os
import pandas as pd

CKPT_DIR = '/content/drive/MyDrive/Ketastasia/data/pipeline2_checkpoints'

bad_files = []
for f in sorted(glob.glob(f'{CKPT_DIR}/chunk_*.csv')):
    try:
        pd.read_csv(f, usecols=['video_id'])
    except Exception as e:
        bad_files.append((f, os.path.getsize(f), str(e)))

print(f"Found {len(bad_files)} unreadable files out of {len(glob.glob(f'{CKPT_DIR}/chunk_*.csv'))}")

Found 72 unreadable files out of 72


In [9]:
QUARANTINE_DIR = f'{CKPT_DIR}/_corrupted'
os.makedirs(QUARANTINE_DIR, exist_ok=True)

for f, size, err in bad_files:
    os.rename(f, os.path.join(QUARANTINE_DIR, os.path.basename(f)))

print(f"Quarantined {len(bad_files)} files")
print(f"Remaining valid chunks: {len(glob.glob(f'{CKPT_DIR}/chunk_*.csv'))}")

Quarantined 72 files
Remaining valid chunks: 0


In [10]:
def get_processed_ids():
    done = set()
    for f in glob.glob(f'{CKPT_DIR}/chunk_*.csv'):
        if os.path.getsize(f) == 0:
            continue
        try:
            done.update(pd.read_csv(f, usecols=['video_id'])['video_id'].unique())
        except pd.errors.EmptyDataError:
            continue
        except Exception as e:
            print(f"Warning: could not read {f}: {e}")
            continue
    return done

CHECKPOINT_EVERY = 25

processed = get_processed_ids()
print(f"Already processed: {len(processed)} videos — resuming")

remaining = manifest_df[~manifest_df['video_id'].isin(processed)]
buffer_rows = []
chunk_idx = len(glob.glob(f'{CKPT_DIR}/chunk_*.csv'))

for i, (_, row) in enumerate(remaining.iterrows()):
    try:
        buffer_rows.extend(process_video(row))
    except Exception as e:
        print(f"Skipped {row['video_id']}: {e}")
        continue

    if (i + 1) % CHECKPOINT_EVERY == 0:
        if buffer_rows:
            pd.DataFrame(buffer_rows).to_csv(f'{CKPT_DIR}/chunk_{chunk_idx}.csv', index=False)
            print(f"Checkpoint saved: chunk_{chunk_idx} ({i+1}/{len(remaining)} videos)")
            chunk_idx += 1
        else:
            print(f"No landmarks extracted in this batch ({i+1}/{len(remaining)} videos) — skipping empty checkpoint")
        buffer_rows = []

if buffer_rows:
    pd.DataFrame(buffer_rows).to_csv(f'{CKPT_DIR}/chunk_{chunk_idx}.csv', index=False)
    print(f"Final checkpoint saved: chunk_{chunk_idx}")

print("Extraction complete")

Already processed: 0 videos — resuming
Checkpoint saved: chunk_0 (25/1813 videos)
Checkpoint saved: chunk_1 (50/1813 videos)
Checkpoint saved: chunk_2 (75/1813 videos)
Checkpoint saved: chunk_3 (100/1813 videos)
Checkpoint saved: chunk_4 (125/1813 videos)
Checkpoint saved: chunk_5 (150/1813 videos)
Checkpoint saved: chunk_6 (175/1813 videos)
Checkpoint saved: chunk_7 (200/1813 videos)
Checkpoint saved: chunk_8 (225/1813 videos)
Checkpoint saved: chunk_9 (250/1813 videos)
Checkpoint saved: chunk_10 (275/1813 videos)
Checkpoint saved: chunk_11 (300/1813 videos)
Checkpoint saved: chunk_12 (325/1813 videos)
Checkpoint saved: chunk_13 (350/1813 videos)
Checkpoint saved: chunk_14 (375/1813 videos)
Checkpoint saved: chunk_15 (400/1813 videos)
Checkpoint saved: chunk_16 (425/1813 videos)
Checkpoint saved: chunk_17 (450/1813 videos)
Checkpoint saved: chunk_18 (475/1813 videos)
Checkpoint saved: chunk_19 (500/1813 videos)
Checkpoint saved: chunk_20 (525/1813 videos)
Checkpoint saved: chunk_21 (5

In [11]:
all_chunks = glob.glob(f'{CKPT_DIR}/chunk_*.csv')
raw_df = pd.concat([pd.read_csv(c) for c in all_chunks], ignore_index=True)
raw_df = raw_df.drop_duplicates(subset=['video_id', 'frame'])

raw_df.to_csv(f'{BASE}/pipeline2A_33kp_raw.csv', index=False)
print(f"Total frames: {len(raw_df)}")
print(raw_df.groupby('split')['label'].value_counts())

Total frames: 73552
split  label              
test   pullup                 1364
       squat                  1105
       clean_and_jerk         1086
       pushup                  954
       bench_press             839
                              ... 
val    leg_raises              164
       tricep_pushdown         154
       jumping_jacks           143
       russian_twist           139
       decline_bench_press      84
Name: count, Length: 78, dtype: int64


In [12]:
print(raw_df[raw_df['split'] == 'train']['label'].value_counts())

label
squat                  6765
clean_and_jerk         5274
pullup                 4420
pushup                 4331
plank                  2896
bench_press            2643
situp                  1988
lateral_raise          1979
t_bar_row              1816
tricep_dips            1699
bicep_curl             1620
lat_pulldown           1440
incline_bench_press    1401
leg_raises             1396
tricep_pushdown        1365
leg_extension          1357
hammer_curl            1303
shoulder_press         1233
russian_twist          1189
romanian_deadlift      1142
deadlift               1081
chest_fly              1046
hip_thrust             1010
jumping_jacks           819
jump_rope               769
decline_bench_press     619
Name: count, dtype: int64


In [13]:
POSE_LANDMARKS = {
    'nose': 0, 'left_shoulder': 11, 'right_shoulder': 12,
    'left_elbow': 13, 'right_elbow': 14, 'left_wrist': 15, 'right_wrist': 16,
    'left_hip': 23, 'right_hip': 24, 'left_knee': 25, 'right_knee': 26,
    'left_ankle': 27, 'right_ankle': 28,
    'left_heel': 29, 'right_heel': 30,
    'left_foot_index': 31, 'right_foot_index': 32,
}

with open(f'{BASE}/pipeline2A_landmark_config.json', 'w') as f:
    json.dump(POSE_LANDMARKS, f, indent=2)

In [14]:
x_cols = [f'x{i}' for i in range(33)]
y_cols = [f'y{i}' for i in range(33)]

def normalize_pose_batch_33(df):
    xs = df[x_cols].values.astype(float)
    ys = df[y_cols].values.astype(float)

    lh, rh = POSE_LANDMARKS['left_hip'], POSE_LANDMARKS['right_hip']
    ls = POSE_LANDMARKS['left_shoulder']

    center_x = (xs[:, lh] + xs[:, rh]) / 2
    center_y = (ys[:, lh] + ys[:, rh]) / 2
    scale = np.sqrt((xs[:, ls] - xs[:, lh])**2 + (ys[:, ls] - ys[:, lh])**2) + 1e-8

    df[x_cols] = (xs - center_x[:, None]) / scale[:, None]
    df[y_cols] = (ys - center_y[:, None]) / scale[:, None]
    return df

train_df = normalize_pose_batch_33(raw_df[raw_df['split'] == 'train'].copy())
val_df   = normalize_pose_batch_33(raw_df[raw_df['split'] == 'val'].copy())
test_df  = normalize_pose_batch_33(raw_df[raw_df['split'] == 'test'].copy())
print("Normalization done for all splits")

Normalization done for all splits


In [15]:
def calculate_angle(ax, ay, bx, by, cx, cy):
    ba = np.array([ax - bx, ay - by])
    bc = np.array([cx - bx, cy - by])
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))

ANGLE_TRIPLETS = {
    'left_knee':  ('left_hip', 'left_knee', 'left_ankle'),
    'right_knee': ('right_hip', 'right_knee', 'right_ankle'),
    'left_elbow': ('left_shoulder', 'left_elbow', 'left_wrist'),
    'right_elbow':('right_shoulder', 'right_elbow', 'right_wrist'),
    'left_hip':   ('left_shoulder', 'left_hip', 'left_knee'),
    'right_hip':  ('right_shoulder', 'right_hip', 'right_knee'),
}

def extract_angles_33(df):
    for name, (a, b, c) in ANGLE_TRIPLETS.items():
        ia, ib, ic = POSE_LANDMARKS[a], POSE_LANDMARKS[b], POSE_LANDMARKS[c]
        df[f'angle_{name}'] = df.apply(
            lambda r: calculate_angle(r[f'x{ia}'], r[f'y{ia}'],
                                       r[f'x{ib}'], r[f'y{ib}'],
                                       r[f'x{ic}'], r[f'y{ic}']),
            axis=1
        )
    return df

train_df = extract_angles_33(train_df)
val_df   = extract_angles_33(val_df)
test_df  = extract_angles_33(test_df)

angle_cols = [f'angle_{k}' for k in ANGLE_TRIPLETS]
for col in angle_cols:
    for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
        n_bad = d[col].isna().sum()
        if n_bad > 0:
            d[col] = d[col].fillna(d.groupby('label')[col].transform('median'))
print("Angle features done")

/tmp/ipykernel_7452/1679106439.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'angle_{name}'] = df.apply(
/tmp/ipykernel_7452/1679106439.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'angle_{name}'] = df.apply(
/tmp/ipykernel_7452/1679106439.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = f

Angle features done


/tmp/ipykernel_7452/1679106439.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'angle_{name}'] = df.apply(


In [16]:
print(train_df.shape)
print([c for c in train_df.columns if 'angle' in c])

(52601, 110)
['angle_left_knee', 'angle_right_knee', 'angle_left_elbow', 'angle_right_elbow', 'angle_left_hip', 'angle_right_hip']


In [17]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.utils import to_categorical

le = LabelEncoder()
train_df['label_encoded'] = le.fit_transform(train_df['label'])
val_df['label_encoded']   = le.transform(val_df['label'])
test_df['label_encoded']  = le.transform(test_df['label'])

feature_cols = x_cols + y_cols + angle_cols  # 66 + 6 = 72 features

def build_sequences(df, seq_len=15, step=7):
    X_list, y_list = [], []
    for video_id, group in df.groupby('video_id'):
        group = group.sort_values('frame')
        feats = group[feature_cols].values
        label = group['label_encoded'].iloc[0]
        if len(feats) < seq_len:
            continue
        for start in range(0, len(feats) - seq_len + 1, step):
            X_list.append(feats[start:start + seq_len])
            y_list.append(label)
    return np.array(X_list), np.array(y_list)

X_tr_raw, y_tr_raw   = build_sequences(train_df)
X_val_raw, y_val_raw = build_sequences(val_df)
X_test_raw, y_test_raw = build_sequences(test_df)

print(f"X_train: {X_tr_raw.shape} | X_val: {X_val_raw.shape} | X_test: {X_test_raw.shape}")

/tmp/ipykernel_7452/682796085.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df['label_encoded'] = le.fit_transform(train_df['label'])
/tmp/ipykernel_7452/682796085.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  val_df['label_encoded']   = le.transform(val_df['label'])
/tmp/ipykernel_7452/682796085.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) inst

X_train: (5579, 15, 72) | X_val: (1102, 15, 72) | X_test: (1100, 15, 72)


In [18]:
n_timesteps, n_features = X_tr_raw.shape[1], X_tr_raw.shape[2]
scaler = StandardScaler()

X_train = scaler.fit_transform(X_tr_raw.reshape(-1, n_features)).reshape(X_tr_raw.shape)
X_val   = scaler.transform(X_val_raw.reshape(-1, n_features)).reshape(X_val_raw.shape)
X_test  = scaler.transform(X_test_raw.reshape(-1, n_features)).reshape(X_test_raw.shape)

num_classes = len(le.classes_)

y_train = to_categorical(y_tr_raw, num_classes=num_classes)   # one-hot
y_val   = to_categorical(y_val_raw, num_classes=num_classes)
y_test  = to_categorical(y_test_raw, num_classes=num_classes)

y_train_int = y_tr_raw   # integer labels
y_val_int   = y_val_raw
y_test_int  = y_test_raw

save_path = f'{BASE}/dataset_seq_33kp_ready.npz'
np.savez(save_path,
         X_train=X_train, y_train=y_train, y_train_int=y_train_int,
         X_val=X_val,     y_val=y_val,     y_val_int=y_val_int,
         X_test=X_test,   y_test=y_test,   y_test_int=y_test_int)

import joblib
joblib.dump(scaler, f'{BASE}/pipeline2A_scaler.pkl')
joblib.dump(le, f'{BASE}/pipeline2A_label_encoder.pkl')

meta = {
    "classes": list(le.classes_),
    "n_keypoints": 33,
    "n_features_per_frame": n_features,
    "feature_order": feature_cols,
    "sequence_length": n_timesteps,
    "extraction_stride": STRIDE,
    "normalization": "hip_centered_shoulder_hip_scaled",
    "sources_extracted_with": "mediapipe_pose_landmarker_full (both Penn Action and Kaggle)",
    "final_counts": {
        "train_sequences": int(X_train.shape[0]),
        "val_sequences": int(X_val.shape[0]),
        "test_sequences": int(X_test.shape[0]),
    }
}
with open(f'{BASE}/pipeline2A_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

print("Pipeline 2 complete — saved:", save_path)

Pipeline 2 complete — saved: /content/drive/MyDrive/Ketastasia/data/dataset_seq_33kp_ready.npz
